# Student Performance Prediction System
## A Dual Machine Learning Framework for Exam Score Regression and Tier Classification

**Course / Assessment:** Machine Learning & Predictive Analytics  
**Dataset:** Cortez and Silva (2008), *Student Performance Dataset*, UCI Machine Learning Repository  

---

### Project Objectives:
1. **Regression Task:** Predict secondary school students' final exam grades ($G3 \in [0, 20]$) using demographic, social, and prior academic features.
2. **Classification Task:** Classify students into actionable academic performance categories:
   - **Low:** $0 \le G3 < 10$ (At-Risk / In need of intervention)
   - **Medium:** $10 \le G3 \le 13$ (Satisfactory / Average)
   - **High:** $14 \le G3 \le 20$ (High Achievement / Advanced)
3. **Algorithms Evaluated:**
   - Simple Linear Regression, Multiple Linear Regression, Polynomial Regression (Degree 2)
   - Ridge & Lasso Regularized Regression
   - Support Vector Machines (Linear, Polynomial, and RBF kernels)
   - Decision Tree Regressor & Classifier
   - Random Forest Regressor & Classifier
4. **Validation & Tuning:** 5-Fold Cross-Validation and Exhaustive Grid Search Hyperparameter Tuning (GridSearchCV).


## 1. Environment Setup & Library Imports


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR, SVC
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Plotting settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline
print("All required libraries imported successfully!")


## 2. Dataset Loading & Exploratory Data Analysis (EDA)
The dataset comprises 395 secondary students with 33 attributes spanning academic records, study habits, parental status, and alcohol consumption.


In [ ]:
# Load dataset
data_path = os.path.join('..', 'data', 'raw', 'student_data.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'raw', 'student_data.csv')

df = pd.read_csv(data_path)
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# Check data types and verify zero missing values
print("Dataset Information:")
df.info()
print("\nMissing Values Count per Column:")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# Summary statistics for numerical variables
df.describe().round(2)


### 2.1 Distribution of the Target Variable (G3) and Performance Categories


In [ ]:
def categorize_score(score):
    if score < 10:
        return 'Low'
    elif score <= 13:
        return 'Medium'
    else:
        return 'High'

df['performance_tier'] = df['G3'].apply(categorize_score)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score Histogram + KDE
sns.histplot(df['G3'], kde=True, bins=20, color='#2b5c8f', ax=axes[0], edgecolor='black')
axes[0].axvline(df['G3'].mean(), color='#e74c3c', linestyle='--', label=f"Mean: {df['G3'].mean():.2f}")
axes[0].axvline(df['G3'].median(), color='#27ae60', linestyle=':', label=f"Median: {df['G3'].median():.2f}")
axes[0].set_title('Distribution of Final Exam Scores (G3)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Final Score (0 - 20)')
axes[0].legend()

# Performance Category Distribution
order = ['Low', 'Medium', 'High']
palette = {'Low': '#e74c3c', 'Medium': '#f39c12', 'High': '#27ae60'}
sns.countplot(data=df, x='performance_tier', order=order, palette=palette, ax=axes[1], edgecolor='black')
axes[1].set_title('Student Counts across Academic Performance Tiers', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Performance Category')

plt.tight_layout()
plt.show()

print("Academic Category Proportions:")
print(df['performance_tier'].value_counts(normalize=True).round(3) * 100)


### 2.2 Correlation Analysis: Prior Exam Performance & Behavioral Features


In [ ]:
numeric_cols = [
    'age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures',
    'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences',
    'G1', 'G2', 'G3'
]

corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmin=-1.0, vmax=1.0, annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Correlation Heatmap: Numerical Features vs Exam Performance', fontsize=13, fontweight='bold')
plt.show()


### 2.3 Key Domain Insights: Study Time, Failures, and Attendance


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Weekly Study Time vs G3
study_labels = ['<2 hrs', '2-5 hrs', '5-10 hrs', '>10 hrs']
sns.boxplot(data=df, x='studytime', y='G3', palette='Blues', ax=axes[0])
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(study_labels)
axes[0].set_title('Weekly Study Time vs Final Exam Score', fontweight='bold')

# 2. Course Failures vs G3
sns.boxplot(data=df, x='failures', y='G3', palette='Reds', ax=axes[1])
axes[1].set_title('Past Class Failures vs Final Exam Score', fontweight='bold')

# 3. Absences vs G3
sns.regplot(data=df, x='absences', y='G3', color='#2c3e50', scatter_kws={'alpha': 0.5}, line_kws={'color': '#e74c3c'}, ax=axes[2])
axes[2].set_title('School Absences vs Final Exam Score', fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Data Preprocessing & Feature Pipeline
Steps:
1. Target separation ($y_{\text{reg}} = G3$, $y_{\text{clf}} = \text{performance\_tier}$).
2. Binary mapping ($13$ binary attributes converted to $0/1$).
3. One-hot encoding of nominal variables (`Mjob`, `Fjob`, `reason`, `guardian`).
4. Stratified Train-Test Split ($80\% / 20\%$, `random_state=42`).
5. Feature scaling using `StandardScaler` (critical for SVR, Ridge, Lasso, and Polynomial Regression).


In [ ]:
# Target definitions
y_reg = df['G3'].astype(float)
y_clf = df['G3'].apply(categorize_score)
X_raw = df.drop(columns=['G3', 'performance_tier'])

# Binary variable mapping
binary_map = {
    'school': {'GP': 0, 'MS': 1}, 'sex': {'F': 0, 'M': 1},
    'address': {'R': 0, 'U': 1}, 'famsize': {'LE3': 0, 'GT3': 1},
    'Pstatus': {'A': 0, 'T': 1}, 'schoolsup': {'no': 0, 'yes': 1},
    'famsup': {'no': 0, 'yes': 1}, 'paid': {'no': 0, 'yes': 1},
    'activities': {'no': 0, 'yes': 1}, 'nursery': {'no': 0, 'yes': 1},
    'higher': {'no': 0, 'yes': 1}, 'internet': {'no': 0, 'yes': 1},
    'romantic': {'no': 0, 'yes': 1}
}

X_encoded = X_raw.copy()
for col, mapping in binary_map.items():
    if col in X_encoded.columns:
        X_encoded[col] = X_encoded[col].map(mapping)

nominal_cols = ['Mjob', 'Fjob', 'reason', 'guardian']
X_encoded = pd.get_dummies(X_encoded, columns=nominal_cols, drop_first=True, dtype=int)
feature_names = list(X_encoded.columns)
print(f"Encoded feature matrix shape: {X_encoded.shape}")


In [ ]:
# Stratified Train-Test Split (80% train, 20% test)
X_train_raw, X_test_raw, y_train_reg, y_test_reg, y_train_clf, y_test_clf = train_test_split(
    X_encoded, y_reg, y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

# Feature Scaling with StandardScaler
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=feature_names, index=X_train_raw.index)
X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_names, index=X_test_raw.index)

print(f"Training set samples: {X_train.shape[0]}")
print(f"Test set samples:     {X_test.shape[0]}")


## 4. Regression Modeling & Evaluation
We implement and benchmark 10 regression algorithms:
1. **Simple Linear Regression** ($G2 \to G3$)
2. **Multiple Linear Regression** (All 41 features)
3. **Polynomial Regression** (Degree 2 on $G1, G2$)
4. **Ridge Regression** ($L_2$ Regularization with $\alpha$ tuning)
5. **Lasso Regression** ($L_1$ Regularization with $\alpha$ tuning)
6. **Support Vector Regressor (SVR) - Linear Kernel**
7. **Support Vector Regressor (SVR) - Polynomial Kernel**
8. **Support Vector Regressor (SVR) - RBF Kernel**
9. **Decision Tree Regressor**
10. **Random Forest Regressor**


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
reg_results = []
trained_regressors = {}

def evaluate_reg(name, model, X_tr, X_te, y_tr, y_te, best_params):
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf, scoring='r2')
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    
    mae = mean_absolute_error(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_te, y_pred)
    
    trained_regressors[name] = (model, y_pred)
    reg_results.append({
        'Model': name,
        'Test_R2': round(r2, 4),
        'Test_RMSE': round(rmse, 4),
        'Test_MAE': round(mae, 4),
        'CV_R2_Mean': round(cv_scores.mean(), 4),
        'CV_R2_Std': round(cv_scores.std(), 4),
        'Best_Params': best_params
    })
    print(f"{name:35} | Test R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | 5-Fold CV: {cv_scores.mean():.4f}")


In [ ]:
# 1. Simple Linear Regression (using G2 as primary predictor)
slr = LinearRegression()
evaluate_reg("Simple Linear Regression", slr, X_train[['G2']], X_test[['G2']], y_train_reg, y_test_reg, {'predictor': 'G2'})

# 2. Multiple Linear Regression
mlr = LinearRegression()
evaluate_reg("Multiple Linear Regression", mlr, X_train, X_test, y_train_reg, y_test_reg, {'features': 'all_41'})

# 3. Polynomial Regression (Degree 2)
poly_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('linreg', LinearRegression())
])
evaluate_reg("Polynomial Regression (Degree 2)", poly_pipe, X_train[['G1', 'G2']], X_test[['G1', 'G2']], y_train_reg, y_test_reg, {'degree': 2, 'features': ['G1', 'G2']})

# 4. Ridge Regression (L2) with GridSearchCV
grid_ridge = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]}, cv=kf, scoring='r2')
grid_ridge.fit(X_train, y_train_reg)
evaluate_reg("Ridge Regression", grid_ridge.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_ridge.best_params_)

# 5. Lasso Regression (L1) with GridSearchCV
grid_lasso = GridSearchCV(Lasso(max_iter=10000, random_state=42), {'alpha': [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]}, cv=kf, scoring='r2')
grid_lasso.fit(X_train, y_train_reg)
evaluate_reg("Lasso Regression", grid_lasso.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_lasso.best_params_)

# 6. SVR (Linear Kernel)
grid_svr_lin = GridSearchCV(SVR(kernel='linear'), {'C': [0.1, 1.0, 10.0], 'epsilon': [0.05, 0.1, 0.2]}, cv=kf, scoring='r2')
grid_svr_lin.fit(X_train, y_train_reg)
evaluate_reg("SVR (Linear Kernel)", grid_svr_lin.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_svr_lin.best_params_)

# 7. SVR (Polynomial Kernel)
grid_svr_poly = GridSearchCV(SVR(kernel='poly'), {'degree': [2, 3], 'C': [0.1, 1.0, 10.0], 'epsilon': [0.05, 0.1, 0.2]}, cv=kf, scoring='r2')
grid_svr_poly.fit(X_train, y_train_reg)
evaluate_reg("SVR (Polynomial Kernel)", grid_svr_poly.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_svr_poly.best_params_)

# 8. SVR (RBF Kernel)
grid_svr_rbf = GridSearchCV(SVR(kernel='rbf'), {'C': [0.1, 1.0, 10.0, 50.0], 'gamma': ['scale', 0.01, 0.1], 'epsilon': [0.05, 0.1, 0.2]}, cv=kf, scoring='r2')
grid_svr_rbf.fit(X_train, y_train_reg)
evaluate_reg("SVR (RBF Kernel)", grid_svr_rbf.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_svr_rbf.best_params_)

# 9. Decision Tree Regressor
grid_dt_reg = GridSearchCV(DecisionTreeRegressor(random_state=42), {'max_depth': [3, 5, 7, None], 'min_samples_split': [2, 5, 10]}, cv=kf, scoring='r2')
grid_dt_reg.fit(X_train, y_train_reg)
evaluate_reg("Decision Tree Regressor", grid_dt_reg.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_dt_reg.best_params_)

# 10. Random Forest Regressor
grid_rf_reg = GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 5]}, cv=kf, scoring='r2')
grid_rf_reg.fit(X_train, y_train_reg)
evaluate_reg("Random Forest Regressor", grid_rf_reg.best_estimator_, X_train, X_test, y_train_reg, y_test_reg, grid_rf_reg.best_params_)


In [ ]:
# Display Regression Comparison Leaderboard
df_reg_summary = pd.DataFrame(reg_results).sort_values(by='Test_R2', ascending=False).reset_index(drop=True)
df_reg_summary[['Model', 'Test_R2', 'Test_RMSE', 'Test_MAE', 'CV_R2_Mean']]


### 4.1 Actual vs. Predicted & Residual Visualizations


In [ ]:
best_reg_name = df_reg_summary.iloc[0]['Model']
best_reg_model, best_reg_pred = trained_regressors[best_reg_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted
axes[0].scatter(y_test_reg, best_reg_pred, color='#2980b9', edgecolors='k', s=60, alpha=0.7)
min_v = min(min(y_test_reg), min(best_reg_pred)) - 1
max_v = max(max(y_test_reg), max(best_reg_pred)) + 1
axes[0].plot([min_v, max_v], [min_v, max_v], color='#e74c3c', linestyle='--', linewidth=2, label='Ideal y = x')
axes[0].set_title(f"Actual vs Predicted ({best_reg_name})\nR²: {df_reg_summary.iloc[0]['Test_R2']:.4f}", fontweight='bold')
axes[0].set_xlabel('Actual Final Exam Grade (G3)')
axes[0].set_ylabel('Predicted Final Exam Grade')
axes[0].legend()

# Residuals vs Predicted
residuals = y_test_reg - best_reg_pred
axes[1].scatter(best_reg_pred, residuals, color='#8e44ad', edgecolors='k', s=60, alpha=0.7)
axes[1].axhline(0, color='#e74c3c', linestyle='--', linewidth=2)
axes[1].set_title(f"Residual Errors vs Predicted ({best_reg_name})", fontweight='bold')
axes[1].set_xlabel('Predicted Score')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()


## 5. Classification Modeling & Evaluation
We classify students into three performance tiers:
- **Low:** $0 \le G3 < 10$
- **Medium:** $10 \le G3 \le 13$
- **High:** $14 \le G3 \le 20$

Algorithms evaluated:
1. **SVC (Linear Kernel)**
2. **SVC (Polynomial Kernel)**
3. **SVC (RBF Kernel)**
4. **Decision Tree Classifier**
5. **Random Forest Classifier**


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
clf_results = []
trained_classifiers = {}
class_labels = ['Low', 'Medium', 'High']

def evaluate_clf(name, model, best_params):
    cv_scores = cross_val_score(model, X_train, y_train_clf, cv=skf, scoring='accuracy')
    model.fit(X_train, y_train_clf)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test_clf, y_pred)
    f1_m = f1_score(y_test_clf, y_pred, average='macro', zero_division=0)
    p_m = precision_score(y_test_clf, y_pred, average='macro', zero_division=0)
    r_m = recall_score(y_test_clf, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_test_clf, y_pred, labels=class_labels)
    
    trained_classifiers[name] = (model, y_pred, cm)
    clf_results.append({
        'Model': name,
        'Test_Accuracy': round(acc, 4),
        'F1_Macro': round(f1_m, 4),
        'Precision_Macro': round(p_m, 4),
        'Recall_Macro': round(r_m, 4),
        'CV_Accuracy_Mean': round(cv_scores.mean(), 4),
        'CV_Accuracy_Std': round(cv_scores.std(), 4),
        'Best_Params': best_params
    })
    print(f"{name:28} | Test Acc: {acc*100:.2f}% | F1-Macro: {f1_m:.4f} | 5-Fold CV: {cv_scores.mean()*100:.2f}%")


In [ ]:
# 1. SVC (Linear Kernel)
grid_svc_lin = GridSearchCV(SVC(kernel='linear', probability=True, random_state=42), {'C': [0.1, 1.0, 10.0]}, cv=skf, scoring='accuracy')
grid_svc_lin.fit(X_train, y_train_clf)
evaluate_clf("SVC (Linear Kernel)", grid_svc_lin.best_estimator_, grid_svc_lin.best_params_)

# 2. SVC (Polynomial Kernel)
grid_svc_poly = GridSearchCV(SVC(kernel='poly', probability=True, random_state=42), {'degree': [2, 3], 'C': [0.1, 1.0, 10.0]}, cv=skf, scoring='accuracy')
grid_svc_poly.fit(X_train, y_train_clf)
evaluate_clf("SVC (Polynomial Kernel)", grid_svc_poly.best_estimator_, grid_svc_poly.best_params_)

# 3. SVC (RBF Kernel)
grid_svc_rbf = GridSearchCV(SVC(kernel='rbf', probability=True, random_state=42), {'C': [0.1, 1.0, 10.0, 20.0], 'gamma': ['scale', 0.01, 0.1]}, cv=skf, scoring='accuracy')
grid_svc_rbf.fit(X_train, y_train_clf)
evaluate_clf("SVC (RBF Kernel)", grid_svc_rbf.best_estimator_, grid_svc_rbf.best_params_)

# 4. Decision Tree Classifier
grid_dt_clf = GridSearchCV(DecisionTreeClassifier(random_state=42), {'criterion': ['gini', 'entropy'], 'max_depth': [3, 5, 8, None], 'min_samples_split': [2, 5, 10]}, cv=skf, scoring='accuracy')
grid_dt_clf.fit(X_train, y_train_clf)
evaluate_clf("Decision Tree Classifier", grid_dt_clf.best_estimator_, grid_dt_clf.best_params_)

# 5. Random Forest Classifier
grid_rf_clf = GridSearchCV(RandomForestClassifier(random_state=42), {'n_estimators': [50, 100, 150], 'max_depth': [5, 10, None], 'min_samples_split': [2, 5]}, cv=skf, scoring='accuracy')
grid_rf_clf.fit(X_train, y_train_clf)
evaluate_clf("Random Forest Classifier", grid_rf_clf.best_estimator_, grid_rf_clf.best_params_)


In [ ]:
# Display Classification Comparison Leaderboard
df_clf_summary = pd.DataFrame(clf_results).sort_values(by='Test_Accuracy', ascending=False).reset_index(drop=True)
df_clf_summary[['Model', 'Test_Accuracy', 'F1_Macro', 'Precision_Macro', 'Recall_Macro', 'CV_Accuracy_Mean']]


### 5.1 Confusion Matrices for All Classifiers


In [ ]:
fig, axes = plt.subplots(1, len(trained_classifiers), figsize=(22, 4.5))

for idx, (name, (model, y_pred, cm)) in enumerate(trained_classifiers.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=class_labels, yticklabels=class_labels, ax=axes[idx],
                linewidths=1, linecolor='white')
    acc = accuracy_score(y_test_clf, y_pred)
    axes[idx].set_title(f"{name}\nAcc: {acc*100:.1f}%", fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    if idx == 0:
        axes[idx].set_ylabel('Actual')
    else:
        axes[idx].set_ylabel('')

plt.tight_layout()
plt.show()


### 5.2 Feature Importance Analysis (Tree Models)


In [ ]:
dt_model = trained_classifiers["Decision Tree Classifier"][0]
rf_model = trained_classifiers["Random Forest Classifier"][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Decision Tree Importance
df_dt_imp = pd.DataFrame({'Feature': feature_names, 'Importance': dt_model.feature_importances_}).sort_values(by='Importance', ascending=False).head(10)
sns.barplot(data=df_dt_imp, x='Importance', y='Feature', palette='mako', ax=axes[0])
axes[0].set_title('Decision Tree: Top 10 Features (Entropy Split)', fontweight='bold')

# Random Forest Importance
df_rf_imp = pd.DataFrame({'Feature': feature_names, 'Importance': rf_model.feature_importances_}).sort_values(by='Importance', ascending=False).head(10)
sns.barplot(data=df_rf_imp, x='Importance', y='Feature', palette='viridis', ax=axes[1])
axes[1].set_title('Random Forest: Top 10 Features (MDI Impurity)', fontweight='bold')

plt.tight_layout()
plt.show()


## 6. Academic Conclusions & Real-World Implications

### Key Findings:
1. **Best Regression Model:**  
   - **Random Forest Regressor** achieved the highest generalization performance with **$R^2 = 0.8737$**, **$\text{RMSE} = 1.745$**, and **$\text{MAE} = 1.079$**.  
   - Ensembling multiple decorrelated decision trees effectively eliminated high variance while capturing subtle non-linear interactions between prior scores, study hours, and absences.
2. **Best Classification Model:**  
   - **Decision Tree Classifier** attained a test accuracy of **$89.87\%$** and an **$F_1\text{-Macro}$ of $0.9070$**, closely followed by **SVC with Linear Kernel ($87.34\%$)**.  
   - The tree identified clear, interpretable decision thresholds primarily driven by period grades ($G2, G1$) and past failures.
3. **Core Predictive Determinants:**  
   - **Period 2 Grade ($G2$):** Possesses the strongest linear correlation ($r = 0.90$) with final exam performance.
   - **Past Course Failures:** The strongest negative indicator; students with $\ge 1$ failures exhibited a dramatic decline in the probability of reaching the High achievement tier.
   - **Weekly Study Time & Absences:** Behavioral features that significantly shift marginal students between the Low (At-Risk) and Medium (Pass) tiers.

### Recommended Educational Interventions:
- **Early Warning Systems:** Flag students whose $G1$ score is below 10 for automated tutorial assignments before $G2$.
- **Absence Threshold Alerts:** Trigger counseling sessions when absences exceed 10 days.
- **Resource Allocation:** Deploy supplementary mathematics labs specifically targeting students identified in the Low tier.
